In [ ]:
from datasets import Dataset, DatasetDict
import random
from math import floor
from pathlib import Path
from conllu import parse

def read_conllu(path):
    with open(path, "r", encoding="utf-8") as f:
      return parse(f.read())

# File lives at ./data/haw_pos_corpus.conllu
conllu_path = Path("data/haw_pos_corpus.conllu")
sentences = read_conllu(conllu_path)

print(f"✅ Loaded {len(sentences)} sentences from {conllu_path}")

✅ Loaded 4 sentences from ../data/haw_pos_corpus.conllu


In [3]:
from datasets import Dataset, DatasetDict
from conllu import parse
import random
from math import floor

def load_conllu_split(path: str, split=(0.8, 0.1, 0.1), seed=42) -> DatasetDict:
    random.seed(seed)

    with open(path, "r", encoding="utf-8") as f:
        sentences = parse(f.read())

    total = len(sentences)
    if total < 3:
        raise ValueError("Not enough sentences to split into train/val/test")

    random.shuffle(sentences)

    # Initial split with floor
    n_train = max(1, floor(split[0] * total))
    n_val = max(1, floor(split[1] * total))
    n_test = max(1, total - n_train - n_val)

    # Fix if overflowed
    overflow = n_train + n_val + n_test - total
    if overflow > 0:
        # Reduce from the largest of the three
        for n in [("train", n_train), ("val", n_val), ("test", n_test)]:
            if overflow == 0:
                break
            name, count = n
            if count > 1:
                reduction = min(overflow, count - 1)
                if name == "train":
                    n_train -= reduction
                elif name == "val":
                    n_val -= reduction
                elif name == "test":
                    n_test -= reduction
                overflow -= reduction

    def to_examples(sentences):
        return [
            {
                "tokens": [t["form"] for t in s],
                "upos": [t["upos"] for t in s]
            }
            for s in sentences
        ]

    train = to_examples(sentences[:n_train])
    val = to_examples(sentences[n_train:n_train + n_val])
    test = to_examples(sentences[n_train + n_val:n_train + n_val + n_test])

    return DatasetDict({
        "train": Dataset.from_list(train),
        "validation": Dataset.from_list(val),
        "test": Dataset.from_list(test),
    })


In [4]:
dataset = load_conllu_split("../data/haw_pos_corpus.conllu")


In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['tokens', 'upos'],
        num_rows: 1
    })
    test: Dataset({
        features: ['tokens', 'upos'],
        num_rows: 1
    })
})

In [6]:
dataset = load_conllu_split("../data/haw_pos_corpus.conllu")
print(dataset)

print("Train sample:", dataset["test"][0])


DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['tokens', 'upos'],
        num_rows: 1
    })
    test: Dataset({
        features: ['tokens', 'upos'],
        num_rows: 1
    })
})
Train sample: {'tokens': ['I', 'love', 'Hawaiian', '.'], 'upos': ['PRON', 'VERB', 'PROPN', 'PUNCT']}


In [ ]:
from datasets import DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    TrainerCallback
)
import evaluate
import numpy as np
import mlflow
import yaml
import torch

# Load config
cfg = yaml.safe_load(open("../params.yaml"))

collator = DataCollatorForTokenClassification(tokenizer=tok)

# Assume dataset is already loaded
# dataset = load_conllu_split("your_file.conllu")

# Step 1: Build tag sets
tag_set   = sorted({tag for split in dataset for row in dataset[split] for tag in row["upos"]})
label2id  = {t: i for i, t in enumerate(tag_set)}
id2label  = {i: t for t, i in label2id.items()}

# Step 2: Encode labels
def encode_labels(example):
    example["labels"] = [label2id[t] for t in example["upos"]]
    return example

dataset = dataset.map(encode_labels)

# Step 3: Tokenize and align labels
tok = AutoTokenizer.from_pretrained("../quick-haw-mlm/checkpoint-300")

def tokenize_and_align(example):
    enc = tok(example["tokens"],
              is_split_into_words=True,
              truncation=True,
              padding="max_length",
              max_length=cfg["max_length"])

    word_ids = enc.word_ids()
    aligned = []
    prev = None
    for wid in word_ids:
        if wid is None:
            aligned.append(-100)
        elif wid != prev:
            aligned.append(example["labels"][wid])
            prev = wid
        else:
            aligned.append(example["labels"][wid])
    enc["labels"] = aligned
    return enc

dataset_tok = dataset.map(tokenize_and_align, batched=False)

# Step 4: Remove original columns
dataset_tok = dataset_tok.remove_columns(["tokens", "upos"])  # ✅ keep "labels"

# Step 5: Load model
model = AutoModelForTokenClassification.from_pretrained(
    cfg["checkpoint_path"],
    num_labels=len(tag_set),
    id2label=id2label,
    label2id=label2id,
)

# Step 6: Define metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred, label in zip(predictions, labels):
        for p, l in zip(pred, label):
            if l != -100:
                true_labels.append(l)
                true_preds.append(p)

    accuracy = np.mean(np.array(true_labels) == np.array(true_preds))
    return {"accuracy": accuracy}

# Step 7: Setup Trainer
args = TrainingArguments(
    output_dir="../model_out/pos",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,

        logging_dir="tb_runs/ewt-bert",
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="epoch",

        report_to="tensorboard",
        seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset_tok["train"],
    eval_dataset=dataset_tok["validation"],
    tokenizer=tok,
    compute_metrics=compute_metrics,
    data_collator=collator,
)

# Step 8: Train and evaluate
trainer.train()
metrics = trainer.evaluate()
metrics

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at ../quick-haw-mlm/checkpoint-300 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_573483/3352601658.py:108: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.787591,0.500000
2,No log,1.778549,0.500000
3,No log,1.774142,0.500000


{'eval_loss': 1.7741416692733765,
 'eval_accuracy': 0.5,
 'eval_runtime': 0.0057,
 'eval_samples_per_second': 175.832,
 'eval_steps_per_second': 175.832,
 'epoch': 3.0}